<a href="https://colab.research.google.com/github/NJerez-dev/Analisis-Datos-Transporte-UM/blob/main/03_gold_kpis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 🥇 Notebook 03 — Capa GOLD + Exports Power BI
**Propósito:** Generar KPIs agregados listos para consumir en Power BI.

| Tabla Gold | KPI Principal |
|---|---|
| `gold_kpi_entregas` | On-Time Delivery %, Lead Time, Retraso promedio |
| `gold_kpi_por_commerce` | Performance por cliente (Falabella / Sodimac) |
| `gold_kpi_por_patente` | Rendimiento por camión/patente |
| `gold_kpi_devoluciones` | Tasa de devolución, motivos principales |
| `gold_fact_viajes` | Tabla de hechos para modelo estrella Power BI |
| `gold_dim_tiempo` | Dimensión calendario |
| `gold_dim_ruta` | Dimensión ruta/zona |

## 0️⃣ Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import sqlite3
import numpy as np
import os
from datetime import datetime, timedelta

DRIVE_BASE   = '/content/drive/MyDrive/SUPPLY_CHAIN_PROJECT'
DB_PATH      = f'{DRIVE_BASE}/supply_chain.db'
EXPORTS_PATH = f'{DRIVE_BASE}/exports'
os.makedirs(EXPORTS_PATH, exist_ok=True)

con = sqlite3.connect(DB_PATH)
print('✅ Conectado a supply_chain.db')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Conectado a supply_chain.db


## 1️⃣ Leer capas Silver

In [ ]:
sv = pd.read_sql('SELECT * FROM silver_viajes', con)
dd = pd.read_sql('SELECT * FROM silver_devoluciones', con)

# Re-convertir fechas
for col in ['fecha_inicio_ruta','fecha_pactada','fecha_entrega_real','fecha_estimada']:
    sv[col] = pd.to_datetime(sv[col], errors='coerce')
dd['fecha_viaje']      = pd.to_datetime(dd['fecha_viaje'],      errors='coerce')
dd['fecha_devolucion'] = pd.to_datetime(dd['fecha_devolucion'], errors='coerce')

# Numéricos
sv['volumen']          = pd.to_numeric(sv['volumen'],          errors='coerce')
sv['dias_retraso']     = pd.to_numeric(sv['dias_retraso'],     errors='coerce')
sv['entrega_on_time']  = pd.to_numeric(sv['entrega_on_time'],  errors='coerce')
sv['flag_no_entregado']= pd.to_numeric(sv['flag_no_entregado'],errors='coerce')

print(f'✅ silver_viajes: {len(sv):,} filas')
print(f'✅ silver_devoluciones: {len(dd):,} filas')

✅ silver_viajes: 1,013 filas
✅ silver_devoluciones: 275 filas


## 2️⃣ Gold — KPI General de Entregas

In [ ]:
# KPI global
total_viajes      = len(sv)
terminados        = (sv['estado'] == 'Terminado').sum()
on_time           = sv['entrega_on_time'].sum()
pct_on_time       = sv['entrega_on_time'].mean() * 100
lead_time_prom    = sv['dias_retraso'].mean()
tasa_devolucion   = len(dd) / total_viajes * 100
rutas_unicas      = sv['id_ruta'].nunique()
patentes_unicas   = sv['patente'].nunique()
volumen_total     = sv['volumen'].sum()

gold_kpi_entregas = pd.DataFrame([{
    'total_viajes'          : total_viajes,
    'viajes_terminados'     : int(terminados),
    'viajes_on_time'        : int(on_time),
    'pct_on_time_delivery'  : round(pct_on_time, 2),
    'pct_no_entregado'      : round(sv['flag_no_entregado'].mean() * 100, 2),
    'lead_time_dias_prom'   : round(lead_time_prom, 2),
    'total_devoluciones'    : len(dd),
    'tasa_devolucion_pct'   : round(tasa_devolucion, 2),
    'rutas_unicas'          : int(rutas_unicas),
    'patentes_activas'      : int(patentes_unicas),
    'volumen_total_m3'      : round(volumen_total, 4),
    '_gold_timestamp'       : datetime.now().isoformat()
}])

print('📊 KPI GENERAL DE ENTREGAS:')
gold_kpi_entregas.T

📊 KPI GENERAL DE ENTREGAS:


,0
total_viajes,1013
viajes_terminados,530
viajes_on_time,1013
pct_on_time_delivery,100.0
pct_no_entregado,47.68
lead_time_dias_prom,0.0
total_devoluciones,275
tasa_devolucion_pct,27.15
rutas_unicas,10
patentes_activas,7


## 3️⃣ Gold — KPI por Commerce (Falabella / Sodimac)

In [ ]:
gold_por_commerce = (
    sv.groupby('commerce')
    .agg(
        total_viajes        = ('suborden',         'count'),
        on_time_count       = ('entrega_on_time',  'sum'),
        pct_on_time         = ('entrega_on_time',  lambda x: round(x.mean()*100, 2)),
        lead_time_prom_dias = ('dias_retraso',      lambda x: round(x.mean(), 2)),
        no_entregados       = ('flag_no_entregado', 'sum'),
        pct_no_entregado    = ('flag_no_entregado', lambda x: round(x.mean()*100, 2)),
        volumen_total       = ('volumen',           lambda x: round(x.sum(), 4)),
        rutas_unicas        = ('id_ruta',           'nunique'),
        patentes_unicas     = ('patente',           'nunique')
    )
    .reset_index()
)

gold_por_commerce['_gold_timestamp'] = datetime.now().isoformat()
print('📊 KPI POR COMMERCE:')
gold_por_commerce

📊 KPI POR COMMERCE:


,commerce,total_viajes,on_time_count,pct_on_time,lead_time_prom_dias,no_entregados,pct_no_entregado,volumen_total,rutas_unicas,patentes_unicas,_gold_timestamp
0,Falabella,1011,1011,100.0,0.0,482,47.68,53.2178,10,7,2026-04-09T01:38:38.058599
1,Sodimac,2,2,100.0,0.0,1,50.00,0.0056,1,1,2026-04-09T01:38:38.058599


## 4️⃣ Gold — KPI por Patente (rendimiento por camión)

In [ ]:
gold_por_patente = (
    sv.groupby('patente')
    .agg(
        total_entregas      = ('suborden',         'count'),
        on_time_count       = ('entrega_on_time',  'sum'),
        pct_on_time         = ('entrega_on_time',  lambda x: round(x.mean()*100, 2)),
        retraso_prom_dias   = ('dias_retraso',      lambda x: round(x.mean(), 2)),
        no_entregados       = ('flag_no_entregado', 'sum'),
        commerce_clientes   = ('commerce',          lambda x: ', '.join(x.unique())),
        regiones_cubiertas  = ('region',            lambda x: ', '.join(x.unique())),
        rutas_asignadas     = ('id_ruta',           'nunique')
    )
    .reset_index()
    .sort_values('pct_on_time', ascending=False)
)

gold_por_patente['_gold_timestamp'] = datetime.now().isoformat()
print(f'📊 KPI POR PATENTE ({len(gold_por_patente)} patentes):')
gold_por_patente.head(10)

📊 KPI POR PATENTE (7 patentes):


,patente,total_entregas,on_time_count,pct_on_time,retraso_prom_dias,no_entregados,commerce_clientes,regiones_cubiertas,rutas_asignadas,_gold_timestamp
0,JKDP54,249,249,100.0,0.0,0,Falabella,Región Metropolitana,1,2026-04-09T01:38:56.092891
1,RDBW74,53,53,100.0,0.0,31,Falabella,Región Metropolitana,1,2026-04-09T01:38:56.092891
2,RSPD18,191,191,100.0,0.0,96,Falabella,Región Metropolitana,2,2026-04-09T01:38:56.092891
3,RTZK81,189,189,100.0,0.0,96,Falabella,Región Metropolitana,2,2026-04-09T01:38:56.092891
4,SSJ070,27,27,100.0,0.0,2,"Falabella, Sodimac",Región de Valparaíso,2,2026-04-09T01:38:56.092891
5,STKY45,154,154,100.0,0.0,126,Falabella,Región Metropolitana,1,2026-04-09T01:38:56.092891
6,STKY72,150,150,100.0,0.0,132,Falabella,Región Metropolitana,1,2026-04-09T01:38:56.092891


## 5️⃣ Gold — KPI Devoluciones

In [ ]:
# Devoluciones por motivo
gold_kpi_dev_motivo = (
    dd.groupby('observaciones')
    .agg(
        cantidad            = ('nro_etiqueta', 'count'),
        dias_devolucion_prom= ('dias_hasta_devolucion', lambda x: round(pd.to_numeric(x, errors='coerce').mean(), 1))
    )
    .reset_index()
    .sort_values('cantidad', ascending=False)
)
gold_kpi_dev_motivo['pct_del_total'] = round(
    gold_kpi_dev_motivo['cantidad'] / gold_kpi_dev_motivo['cantidad'].sum() * 100, 2
)
gold_kpi_dev_motivo['_gold_timestamp'] = datetime.now().isoformat()

print('📊 KPI DEVOLUCIONES POR MOTIVO:')
gold_kpi_dev_motivo

📊 KPI DEVOLUCIONES POR MOTIVO:


,observaciones,cantidad,dias_devolucion_prom,pct_del_total,_gold_timestamp
1,Tienda Errónea,231,0.0,84.0,2026-04-09T01:39:11.005164
0,-,44,1.9,16.0,2026-04-09T01:39:11.005164


## 6️⃣ Gold — Dimensión Tiempo (calendario)

In [ ]:
# Rango de fechas del dataset
fecha_min = sv['fecha_pactada'].min()
fecha_max = sv['fecha_pactada'].max()

fechas = pd.date_range(start=fecha_min, end=fecha_max, freq='D')
gold_dim_tiempo = pd.DataFrame({
    'fecha'          : fechas,
    'anio'           : fechas.year,
    'mes'            : fechas.month,
    'nombre_mes'     : fechas.strftime('%B'),
    'dia'            : fechas.day,
    'dia_semana'     : fechas.dayofweek,
    'nombre_dia'     : fechas.strftime('%A'),
    'semana_anio'    : fechas.isocalendar().week.values,
    'trimestre'      : fechas.quarter,
    'es_fin_semana'  : (fechas.dayofweek >= 5).astype(int),
    'anio_mes'       : fechas.strftime('%Y-%m')
})
gold_dim_tiempo['fecha'] = gold_dim_tiempo['fecha'].astype(str)

print(f'📅 Dimensión tiempo: {len(gold_dim_tiempo)} días ({fecha_min.date()} → {fecha_max.date()})')
gold_dim_tiempo.head(5)

📅 Dimensión tiempo: 1 días (2026-03-02 → 2026-03-02)


,fecha,anio,mes,nombre_mes,dia,dia_semana,nombre_dia,semana_anio,trimestre,es_fin_semana,anio_mes
0,2026-03-02,2026,3,March,2,0,Monday,10,1,0,2026-03


## 7️⃣ Gold — Tabla de Hechos (modelo estrella para Power BI)

In [ ]:
gold_fact_viajes = sv[[
    'suborden','documento','estado','patente','commerce',
    'id_ruta','posicion_ruta','centro_transporte',
    'localidad','region','volumen',
    'fecha_pactada','fecha_entrega_real','fecha_estimada',
    'dias_retraso','entrega_on_time','flag_no_entregado',
    'motivo_no_entrega','activa'
]].copy()

# Claves de fecha para cruzar con dim_tiempo
gold_fact_viajes['fecha_key'] = sv['fecha_pactada'].dt.strftime('%Y-%m-%d')

# Convertir datetime a string para SQLite
for col in ['fecha_pactada','fecha_entrega_real','fecha_estimada']:
    gold_fact_viajes[col] = gold_fact_viajes[col].astype(str)

gold_fact_viajes['_gold_timestamp'] = datetime.now().isoformat()

print(f'📦 gold_fact_viajes: {len(gold_fact_viajes):,} filas × {gold_fact_viajes.shape[1]} columnas')
gold_fact_viajes.head(3)

📦 gold_fact_viajes: 1,013 filas × 21 columnas


,suborden,documento,estado,patente,commerce,id_ruta,posicion_ruta,centro_transporte,localidad,region,...,fecha_pactada,fecha_entrega_real,fecha_estimada,dias_retraso,entrega_on_time,flag_no_entregado,motivo_no_entrega,activa,fecha_key,_gold_timestamp
0,100000000000000,100000000000000,Terminado,STKY45,Falabella,14130168,0,HUB XD,SANTIAGO CENTRO,Región Metropolitana,...,2026-03-02,2026-03-02,2026-03-02,0,1,0,None,1,2026-03-02,2026-04-09T01:39:40.757245
1,100000000000000,100000000000000,Terminado,STKY45,Falabella,14130168,1,HUB XD,SANTIAGO CENTRO,Región Metropolitana,...,2026-03-02,2026-03-02,2026-03-02,0,1,0,None,1,2026-03-02,2026-04-09T01:39:40.757245
2,100000000000000,100000000000000,Terminado,STKY45,Falabella,14130168,2,HUB XD,SANTIAGO CENTRO,Región Metropolitana,...,2026-03-02,2026-03-02,2026-03-02,0,1,0,None,1,2026-03-02,2026-04-09T01:39:40.757245


## 8️⃣ Escribir todas las tablas Gold en SQLite

In [ ]:
tablas_gold = {
    'gold_kpi_entregas'    : gold_kpi_entregas,
    'gold_kpi_por_commerce': gold_por_commerce,
    'gold_kpi_por_patente' : gold_por_patente,
    'gold_kpi_devoluciones': gold_kpi_dev_motivo,
    'gold_dim_tiempo'      : gold_dim_tiempo,
    'gold_fact_viajes'     : gold_fact_viajes
}

for nombre, df_tabla in tablas_gold.items():
    df_tabla.to_sql(nombre, con, if_exists='replace', index=False)
    print(f'  ✅ {nombre}: {len(df_tabla):,} filas')

con.commit()
print('\n✅ Todas las tablas Gold escritas en SQLite')

  ✅ gold_kpi_entregas: 1 filas
  ✅ gold_kpi_por_commerce: 2 filas
  ✅ gold_kpi_por_patente: 7 filas
  ✅ gold_kpi_devoluciones: 2 filas
  ✅ gold_dim_tiempo: 1 filas
  ✅ gold_fact_viajes: 1,013 filas

✅ Todas las tablas Gold escritas en SQLite


## 9️⃣ Exportar CSVs para Power BI

In [ ]:
print('📤 Exportando CSVs para Power BI...')

for nombre, df_tabla in tablas_gold.items():
    ruta_csv = f'{EXPORTS_PATH}/{nombre}.csv'
    df_tabla.to_csv(ruta_csv, index=False, encoding='utf-8-sig')  # utf-8-sig para Excel/PBI
    print(f'  ✅ {nombre}.csv → {ruta_csv}')

print(f'\n📂 Todos los archivos en: {EXPORTS_PATH}')
print('\n🔌 Para Power BI Desktop:')
print('  1. Abre Power BI Desktop')
print('  2. Inicio → Obtener datos → Texto/CSV')
print('  3. Descarga los CSV desde Google Drive y cárgalos uno por uno')
print('  4. En el modelo de datos, relaciona gold_fact_viajes.fecha_key con gold_dim_tiempo.fecha')
print('  5. Relaciona gold_fact_viajes.commerce con gold_kpi_por_commerce.commerce')

📤 Exportando CSVs para Power BI...
  ✅ gold_kpi_entregas.csv → /content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/exports/gold_kpi_entregas.csv
  ✅ gold_kpi_por_commerce.csv → /content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/exports/gold_kpi_por_commerce.csv
  ✅ gold_kpi_por_patente.csv → /content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/exports/gold_kpi_por_patente.csv
  ✅ gold_kpi_devoluciones.csv → /content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/exports/gold_kpi_devoluciones.csv
  ✅ gold_dim_tiempo.csv → /content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/exports/gold_dim_tiempo.csv
  ✅ gold_fact_viajes.csv → /content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/exports/gold_fact_viajes.csv

📂 Todos los archivos en: /content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/exports

🔌 Para Power BI Desktop:
  1. Abre Power BI Desktop
  2. Inicio → Obtener datos → Texto/CSV
  3. Descarga los CSV desde Google Drive y cárgalos uno por uno
  4. En el modelo de datos, relaciona gold_fact_viajes.fecha_key con gold_dim_tiempo.fecha
  5. Relaci

## 🔟 Resumen Final — Todas las capas

In [ ]:
print('=' * 60)
print('  ARQUITECTURA MEDALLION — RESUMEN FINAL')
print('=' * 60)

todas_tablas = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con
)

for t in todas_tablas['name']:
    n = pd.read_sql(f'SELECT COUNT(*) as total FROM {t}', con).iloc[0,0]
    capa = '🥉 BRONZE' if t.startswith('bronze') else ('🥈 SILVER' if t.startswith('silver') else '🥇 GOLD  ')
    print(f'  {capa} | {t:<35} | {n:>6,} filas')

print('=' * 60)

# KPIs principales
kpi = pd.read_sql('SELECT * FROM gold_kpi_entregas', con).iloc[0]
print(f'\n📊 KPIs PRINCIPALES:')
print(f'  On-Time Delivery    : {kpi["pct_on_time_delivery"]}%')
print(f'  No entregados       : {kpi["pct_no_entregado"]}%')
print(f'  Lead Time promedio  : {kpi["lead_time_dias_prom"]} días')
print(f'  Tasa de devolución  : {kpi["tasa_devolucion_pct"]}%')
print(f'  Rutas únicas        : {kpi["rutas_unicas"]}')
print(f'  Patentes activas    : {kpi["patentes_activas"]}')
print(f'  Volumen total (m³)  : {kpi["volumen_total_m3"]}')

con.close()
print('\n🔒 Conexión cerrada')
print('✅ ¡Arquitectura Medallion completa! CSVs listos para Power BI 🚀')

  ARQUITECTURA MEDALLION — RESUMEN FINAL
  🥉 BRONZE | bronze_devoluciones                 |    275 filas
  🥉 BRONZE | bronze_viajes_diarios               |  1,013 filas
  🥇 GOLD   | gold_dim_tiempo                     |      1 filas
  🥇 GOLD   | gold_fact_viajes                    |  1,013 filas
  🥇 GOLD   | gold_kpi_devoluciones               |      2 filas
  🥇 GOLD   | gold_kpi_entregas                   |      1 filas
  🥇 GOLD   | gold_kpi_por_commerce               |      2 filas
  🥇 GOLD   | gold_kpi_por_patente                |      7 filas
  🥈 SILVER | silver_devoluciones                 |    275 filas
  🥈 SILVER | silver_viajes                       |  1,013 filas

📊 KPIs PRINCIPALES:
  On-Time Delivery    : 100.0%
  No entregados       : 47.68%
  Lead Time promedio  : 0.0 días
  Tasa de devolución  : 27.15%
  Rutas únicas        : 10
  Patentes activas    : 7
  Volumen total (m³)  : 53.2233

🔒 Conexión cerrada
✅ ¡Arquitectura Medallion completa! CSVs listos para Power BI 🚀
